In [0]:
-- 05_load
-- Loads dim_customer and dim_product via SCD Type 2 MERGE, loads dim_channel
-- as a stable lookup, then loads dim_date (no SCD) and fact_orders.
-- fact_orders resolves customer_sk and product_sk with point-in-time joins,
-- and resolves channel_sk with a direct channel_name lookup.
-- Source: northriver_staging (xform_* tables) | Target: northriver_warehouse
-- Run after: 04_transform

-- 1. Load dim_customer (SCD Type 2)
MERGE INTO northriver_warehouse.dim_customer AS target
USING northriver_staging.xform_dim_customer AS source
ON target.customer_id = source.customer_id AND target.is_current = TRUE
WHEN MATCHED AND source.change_type = 'CHANGED' THEN
  UPDATE SET target.effective_to = current_date() - INTERVAL 1 DAY, target.is_current = FALSE;

INSERT INTO northriver_warehouse.dim_customer (customer_id, first_name, last_name, email, region, segment, effective_from, effective_to, is_current)
SELECT 
  customer_id,
  first_name,
  last_name,
  email,
  region,
  segment,
  current_date() AS effective_from,
  NULL AS effective_to,
  TRUE AS is_current
FROM northriver_staging.xform_dim_customer
WHERE change_type IN ('NEW', 'CHANGED');

-- 2. Load dim_product (SCD Type 2)
MERGE INTO northriver_warehouse.dim_product AS target
USING northriver_staging.xform_dim_product AS source
ON target.product_id = source.product_id AND target.is_current = TRUE
WHEN MATCHED AND source.change_type = 'CHANGED' THEN
  UPDATE SET target.effective_to = current_date() - INTERVAL 1 DAY, target.is_current = FALSE;

INSERT INTO northriver_warehouse.dim_product (product_id, product_name, brand, category, subcategory, unit_cost, effective_from, effective_to, is_current)
SELECT 
  product_id,
  product_name,
  brand,
  category,
  subcategory,
  unit_cost,
  current_date() AS effective_from,
  NULL AS effective_to,
  TRUE AS is_current
FROM northriver_staging.xform_dim_product
WHERE change_type IN ('NEW', 'CHANGED');

-- 3. Load dim_channel (Lookup upsert)
MERGE INTO northriver_warehouse.dim_channel AS target
USING northriver_staging.xform_dim_channel AS source
ON target.channel_name = source.channel_name
WHEN NOT MATCHED AND source.change_type = 'NEW' THEN
  INSERT (channel_name) VALUES (source.channel_name);

-- 4. Load fact_orders
MERGE INTO northriver_warehouse.fact_orders AS target
USING (
  SELECT 
    f.order_id,
    f.order_item_id,
    f.order_date,
    dc.customer_sk,
    dp.product_sk,
    dch.channel_sk,
    dd.date_sk,
    f.quantity,
    f.unit_price
  FROM northriver_staging.xform_fact_orders f
  LEFT JOIN northriver_warehouse.dim_customer dc 
    ON f.customer_id = dc.customer_id 
    AND f.order_date BETWEEN dc.effective_from AND COALESCE(dc.effective_to, '9999-12-31')
  LEFT JOIN northriver_warehouse.dim_product dp 
    ON f.product_id = dp.product_id 
    AND f.order_date BETWEEN dp.effective_from AND COALESCE(dp.effective_to, '9999-12-31')
  LEFT JOIN northriver_warehouse.dim_channel dch 
    ON f.channel = dch.channel_name
  LEFT JOIN northriver_warehouse.dim_date dd 
    ON f.order_date = dd.full_date
) AS source
ON target.order_item_id = source.order_item_id
WHEN NOT MATCHED THEN
  INSERT (order_id, order_item_id, customer_sk, product_sk, channel_sk, date_sk, quantity, unit_price)
  VALUES (source.order_id, source.order_item_id, source.customer_sk, source.product_sk, source.channel_sk, source.date_sk, source.quantity, source.unit_price);

  SELECT COUNT(*) FROM northriver_warehouse.dim_customer;

  SELECT COUNT(*) FROM northriver_warehouse.fact_orders;
SELECT SUM(revenue) FROM northriver_warehouse.fact_orders;